# Edge Network Analysis: Centrality, Hubs, and Communities

**This notebook analyzes an existing edge network** created from the edge-centric transformation.

---

## Prerequisites

You should have already run the main edge-centric analysis notebook, which creates:
- `edge_network.graphml` - **Required** (main input for this notebook)
- `edge_statistics.csv` - Optional (pre-computed statistics)
- `edge_fc_matrix.npy` - Optional (adjacency matrix)

---

## What This Notebook Does

1. Load an existing edge network
2. Compute centrality measures (degree, betweenness, closeness, eigenvector, PageRank)
3. Identify hub edges
4. Detect communities
5. Analyze community structure
6. Visualize results
7. Export findings

---

## 1. Setup

In [ ]:
# Install if needed
# !pip install python-igraph pandas numpy matplotlib seaborn scipy

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import igraph as ig
from scipy.stats import entropy, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

print("✓ Libraries loaded")
print(f"igraph version: {ig.__version__}")

✓ Libraries loaded
igraph version: 0.11.9


---
## 2. Load Edge Network

### Option A: Load from GraphML (Recommended)

In [2]:
# Load the edge network from GraphML file
filename = 'edge_network.graphml'  # Change this to your file path

try:
    G_edges = ig.Graph.Read_GraphML(filename)
    print(f"✓ Loaded edge network from: {filename}")
    print(f"\nNetwork Properties:")
    print(f"  Edge nodes (vertices): {G_edges.vcount()}")
    print(f"  Edge-edge connections: {G_edges.ecount()}")
    print(f"  Directed: {G_edges.is_directed()}")
    print(f"  Average degree: {np.mean(G_edges.degree()):.2f}")
    print(f"  Density: {G_edges.density():.4f}")
    
    # Check available attributes
    print(f"\nVertex attributes: {G_edges.vs.attributes()}")
    print(f"Edge attributes: {G_edges.es.attributes()}")
    
except FileNotFoundError:
    print(f"❌ Error: Could not find '{filename}'")
    print("Make sure you've run the main edge-centric analysis notebook first!")

✓ Loaded edge network from: edge_network.graphml

Network Properties:
  Edge nodes (vertices): 1091
  Edge-edge connections: 35344
  Directed: False
  Average degree: 64.79
  Density: 0.0594

Vertex attributes: ['name', 'original_weight', 'source_node', 'target_node', 'id']
Edge attributes: ['connection_type']


### Option B: Load from Edge List (Alternative)

If you don't have a GraphML file, you can reconstruct from edge list + attributes.

In [ ]:
# Uncomment if using edge list instead of GraphML

# # Load edge statistics (contains structure info)
# stats_df = pd.read_csv('edge_statistics.csv')
# 
# # Load adjacency matrix
# eFC = np.load('edge_fc_matrix.npy')
# 
# # Reconstruct graph from adjacency matrix
# G_edges = ig.Graph.Adjacency((eFC > 0).tolist(), mode='undirected')
# 
# # Add attributes from stats_df
# G_edges.vs['name'] = stats_df['edge_label'].tolist()
# if 'original_weight' in stats_df.columns:
#     G_edges.vs['original_weight'] = stats_df['original_weight'].tolist()
# 
# print("✓ Reconstructed graph from edge list and matrix")

---
## 3. Compute Centrality Measures

Calculate various centrality measures for edges in the network.

In [ ]:
print("Computing centrality measures...\n")

# Calculate all centralities
centralities = {
    'degree': np.array(G_edges.degree()),
    'betweenness': np.array(G_edges.betweenness()),
    'closeness': np.array(G_edges.closeness()),
    'eigenvector': np.array(G_edges.eigenvector_centrality()),
    'pagerank': np.array(G_edges.pagerank())
}

# Add to graph attributes
for name, values in centralities.items():
    G_edges.vs[name] = values.tolist()

print("✓ Centrality measures computed\n")

# Display statistics
print("="*70)
print("CENTRALITY STATISTICS")
print("="*70)
print(f"{'Measure':<15} {'Mean':<12} {'Std':<12} {'Min':<12} {'Max':<12}")
print("-"*70)
for name, values in centralities.items():
    print(f"{name:<15} {values.mean():<12.4f} {values.std():<12.4f} "
          f"{values.min():<12.4f} {values.max():<12.4f}")
print("="*70)

### Visualize Centrality Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, values) in enumerate(centralities.items()):
    ax = axes[idx]
    
    # Histogram
    ax.hist(values, bins=50, edgecolor='black', alpha=0.7, color=f'C{idx}')
    
    # Mean line
    ax.axvline(values.mean(), color='red', linestyle='--', 
               linewidth=2, label=f'Mean={values.mean():.2e}')
    
    # Median line
    ax.axvline(np.median(values), color='blue', linestyle=':', 
               linewidth=2, label=f'Median={np.median(values):.2e}')
    
    ax.set_xlabel(name.capitalize(), fontsize=11)
    ax.set_ylabel('Frequency', fontsize=11)
    ax.set_title(f'{name.capitalize()} Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

# Remove empty subplot
fig.delaxes(axes[-1])

plt.tight_layout()
plt.savefig('centrality_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: centrality_distributions.png")

### Correlations Between Centrality Measures

In [ ]:
# Calculate correlation matrix
cent_df = pd.DataFrame(centralities)
corr_matrix = cent_df.corr(method='spearman')

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, ax=ax, cbar_kws={'label': 'Spearman r'})
ax.set_title('Correlations Between Centrality Measures', 
             fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('centrality_correlations.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: centrality_correlations.png")
print("\nInterpretation:")
print("  • High correlation: Measures capture similar properties")
print("  • Low correlation: Measures capture different aspects of importance")

### Top Edges by Each Centrality

In [ ]:
top_k = 10

for name, values in centralities.items():
    print("="*70)
    print(f"TOP {top_k} EDGES BY {name.upper()}")
    print("="*70)
    
    top_idx = np.argsort(values)[-top_k:]
    
    for rank, idx in enumerate(reversed(top_idx), 1):
        edge_name = G_edges.vs[idx]['name'] if 'name' in G_edges.vs.attributes() else f"Edge {idx}"
        print(f"{rank:2d}. {edge_name[:60]}")
        print(f"    {name}: {values[idx]:.4f}")
        
        # Show original weight if available
        if 'original_weight' in G_edges.vs.attributes():
            print(f"    Original weight: {G_edges.vs[idx]['original_weight']:.4f}")
    
    print()

---
## 4. Hub Edge Identification

Hub edges are those with high centrality across multiple measures.

### Method 1: High Degree AND High Betweenness

In [ ]:
# Define thresholds (top 10% for each)
degree_threshold = np.percentile(centralities['degree'], 90)
betweenness_threshold = np.percentile(centralities['betweenness'], 90)

# Identify hubs
is_hub = (centralities['degree'] > degree_threshold) & \
         (centralities['betweenness'] > betweenness_threshold)

hub_indices = np.where(is_hub)[0]
n_hubs = len(hub_indices)

# Add hub status to graph
G_edges.vs['is_hub'] = is_hub.tolist()

print("="*70)
print("HUB EDGE IDENTIFICATION (Method 1: Degree + Betweenness)")
print("="*70)
print(f"Degree threshold (90th percentile): {degree_threshold:.2f}")
print(f"Betweenness threshold (90th percentile): {betweenness_threshold:.2f}")
print(f"\nNumber of hub edges: {n_hubs} ({100*n_hubs/G_edges.vcount():.1f}% of all edges)")

if n_hubs > 0:
    print(f"\nHub edges:")
    for idx in hub_indices[:20]:  # Show first 20
        edge_name = G_edges.vs[idx]['name'] if 'name' in G_edges.vs.attributes() else f"Edge {idx}"
        print(f"  {edge_name[:60]}")
        print(f"    Degree: {centralities['degree'][idx]:.0f}, "
              f"Betweenness: {centralities['betweenness'][idx]:.2f}")

### Visualize Hub Edges

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

# Plot all edges
ax.scatter(centralities['degree'], centralities['betweenness'], 
           alpha=0.4, s=30, c='lightblue', label='All edges', edgecolors='gray', linewidth=0.5)

# Highlight hubs
if n_hubs > 0:
    ax.scatter(centralities['degree'][hub_indices], 
               centralities['betweenness'][hub_indices],
               alpha=0.9, s=200, c='red', marker='*', 
               label=f'Hub edges (n={n_hubs})', edgecolors='darkred', linewidth=2)

# Threshold lines
ax.axvline(degree_threshold, color='red', linestyle='--', alpha=0.5, linewidth=2)
ax.axhline(betweenness_threshold, color='red', linestyle='--', alpha=0.5, linewidth=2)

ax.set_xlabel('Degree', fontsize=13, fontweight='bold')
ax.set_ylabel('Betweenness Centrality', fontsize=13, fontweight='bold')
ax.set_title('Hub Edge Identification\n(High Degree + High Betweenness)', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('hub_edges_identification.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Saved: hub_edges_identification.png")

### Method 2: Composite Hub Score

In [ ]:
# Normalize each centrality to [0,1]
normalized_cent = {}
for name, values in centralities.items():
    if values.max() > 0:
        normalized_cent[name] = (values - values.min()) / (values.max() - values.min())
    else:
        normalized_cent[name] = values

# Composite score (average of normalized centralities)
composite_score = np.mean([
    normalized_cent['degree'],
    normalized_cent['betweenness'],
    normalized_cent['eigenvector']
], axis=0)

# Add to graph
G_edges.vs['composite_score'] = composite_score.tolist()

# Top composite hubs
composite_threshold = np.percentile(composite_score, 95)
composite_hubs = np.where(composite_score > composite_threshold)[0]

print("="*70)
print("HUB EDGE IDENTIFICATION (Method 2: Composite Score)")
print("="*70)
print("Composite score = average of normalized (degree, betweenness, eigenvector)")
print(f"Threshold (95th percentile): {composite_threshold:.3f}")
print(f"\nNumber of composite hubs: {len(composite_hubs)} ({100*len(composite_hubs)/G_edges.vcount():.1f}%)")

print(f"\nTop 15 edges by composite score:")
top_composite = np.argsort(composite_score)[-15:]
for rank, idx in enumerate(reversed(top_composite), 1):
    edge_name = G_edges.vs[idx]['name'] if 'name' in G_edges.vs.attributes() else f"Edge {idx}"
    print(f"{rank:2d}. {edge_name[:60]}")
    print(f"    Composite score: {composite_score[idx]:.3f}")

---
## 5. Community Detection

Detect communities (modules) in the edge network.

In [ ]:
print("Testing community detection methods...\n")

methods = ['multilevel', 'leiden', 'infomap', 'label_propagation']
results = {}

print("="*70)
print(f"{'Method':<20} {'Communities':<12} {'Modularity':<12} {'Sizes (first 5)'}")
print("-"*70)

for method in methods:
    try:
        if method == 'multilevel':
            communities = G_edges.community_multilevel()
        elif method == 'leiden':
            communities = G_edges.community_leiden()
        elif method == 'infomap':
            communities = G_edges.community_infomap()
        elif method == 'label_propagation':
            communities = G_edges.community_label_propagation()
        
        results[method] = communities
        sizes = [len(c) for c in communities]
        print(f"{method:<20} {len(communities):<12} {communities.modularity:<12.4f} {sizes[:5]}")
        
    except Exception as e:
        print(f"{method:<20} Not available - {str(e)[:30]}")

print("="*70)

# Select best method (highest modularity)
if results:
    best_method = max(results.items(), key=lambda x: x[1].modularity)
    communities = best_method[1]
    
    print(f"\n✓ Selected method: {best_method[0]} (highest modularity = {communities.modularity:.4f})")
    
    # Add to graph
    G_edges.vs['community'] = communities.membership
else:
    print("\n❌ No community detection method succeeded")

### Analyze Community Structure

In [ ]:
if results:
    print("="*70)
    print("COMMUNITY STRUCTURE ANALYSIS")
    print("="*70)
    print(f"Number of communities: {len(communities)}")
    print(f"Modularity: {communities.modularity:.4f}")
    print(f"\nCommunity sizes:")
    
    # Sort communities by size
    sizes = [(i, len(c)) for i, c in enumerate(communities)]
    sizes.sort(key=lambda x: x[1], reverse=True)
    
    for comm_idx, size in sizes:
        percentage = 100 * size / G_edges.vcount()
        print(f"  Community {comm_idx:2d}: {size:4d} edges ({percentage:5.1f}%)")
    
    # Community size distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    ax = axes[0]
    comm_sizes = [len(c) for c in communities]
    ax.bar(range(len(comm_sizes)), comm_sizes, color='steelblue', edgecolor='black')
    ax.set_xlabel('Community', fontsize=11)
    ax.set_ylabel('Number of Edges', fontsize=11)
    ax.set_title(f'Community Sizes (n={len(communities)})', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    
    # Pie chart
    ax = axes[1]
    # Only show top 10 communities + "Other"
    if len(comm_sizes) > 10:
        top_10 = sorted(comm_sizes, reverse=True)[:10]
        other = sum(sorted(comm_sizes, reverse=True)[10:])
        plot_sizes = top_10 + [other]
        labels = [f"C{i}" for i in range(10)] + ["Other"]
    else:
        plot_sizes = comm_sizes
        labels = [f"C{i}" for i in range(len(comm_sizes))]
    
    ax.pie(plot_sizes, labels=labels, autopct='%1.1f%%', startangle=90)
    ax.set_title('Community Distribution', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('community_sizes.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Saved: community_sizes.png")

### Hub Nodes Within Communities

In [ ]:
if results:
    print("="*70)
    print("HUB EDGES WITHIN COMMUNITIES")
    print("="*70)
    
    for comm_idx, community in enumerate(communities):
        if len(community) < 5:  # Skip small communities
            continue
        
        # Get betweenness for edges in this community
        comm_betweenness = centralities['betweenness'][community]
        
        # Top 5 edges in this community
        top_in_comm = np.argsort(comm_betweenness)[-5:]
        
        print(f"\nCommunity {comm_idx} ({len(community)} edges):")
        print(f"  Top 5 edges by betweenness:")
        
        for rank, local_idx in enumerate(reversed(top_in_comm), 1):
            global_idx = community[local_idx]
            edge_name = G_edges.vs[global_idx]['name'] if 'name' in G_edges.vs.attributes() else f"Edge {global_idx}"
            print(f"    {rank}. {edge_name[:55]}")
            print(f"       Betweenness: {centralities['betweenness'][global_idx]:.2f}, "
                  f"Degree: {centralities['degree'][global_idx]:.0f}")

### Visualize Community Structure

In [ ]:
if results:
    # Get largest component for visualization
    components = G_edges.components()
    largest_comp = max(components, key=len)
    G_vis = G_edges.subgraph(largest_comp)
    
    print(f"Visualizing largest component: {len(largest_comp)} / {G_edges.vcount()} edges")
    print("Computing layout (this may take a moment)...")
    
    # Calculate layout
    layout = G_vis.layout_fruchterman_reingold()
    
    # Prepare colors
    colors = [communities.membership[i] for i in largest_comp]
    
    # Create visualization
    fig, ax = plt.subplots(figsize=(16, 12))
    
    ig.plot(
        G_vis,
        target=ax,
        layout=layout,
        vertex_size=10,
        vertex_color=colors,
        vertex_frame_width=0.5,
        vertex_frame_color='white',
        edge_width=0.2,
        edge_color='#CCCCCC',
        palette=ig.RainbowPalette(n=len(communities))
    )
    
    ax.set_title(f'Edge Network Communities\n'
                 f'({len(communities)} communities, modularity={communities.modularity:.3f})',
                 fontsize=16, fontweight='bold', pad=20)
    ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('community_network_visualization.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Saved: community_network_visualization.png")

---
## 6. Community-Centrality Relationships

Analyze how centrality relates to community structure.

In [ ]:
if results:
    # Calculate average centrality per community
    comm_centrality = {}
    
    for name in centralities.keys():
        comm_centrality[name] = []
        for community in communities:
            avg = np.mean(centralities[name][community])
            comm_centrality[name].append(avg)
    
    # Create DataFrame
    comm_df = pd.DataFrame(comm_centrality)
    comm_df['community'] = range(len(communities))
    comm_df['size'] = [len(c) for c in communities]
    
    print("="*70)
    print("AVERAGE CENTRALITY BY COMMUNITY")
    print("="*70)
    print(comm_df.sort_values('betweenness', ascending=False).head(10))
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Size vs Betweenness
    ax = axes[0, 0]
    ax.scatter(comm_df['size'], comm_df['betweenness'], s=100, alpha=0.6)
    ax.set_xlabel('Community Size', fontsize=11)
    ax.set_ylabel('Avg Betweenness', fontsize=11)
    ax.set_title('Community Size vs Betweenness', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Size vs Degree
    ax = axes[0, 1]
    ax.scatter(comm_df['size'], comm_df['degree'], s=100, alpha=0.6, color='coral')
    ax.set_xlabel('Community Size', fontsize=11)
    ax.set_ylabel('Avg Degree', fontsize=11)
    ax.set_title('Community Size vs Degree', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Degree vs Betweenness by community
    ax = axes[1, 0]
    ax.scatter(comm_df['degree'], comm_df['betweenness'], 
               s=comm_df['size']*2, alpha=0.6, color='green')
    ax.set_xlabel('Avg Degree', fontsize=11)
    ax.set_ylabel('Avg Betweenness', fontsize=11)
    ax.set_title('Degree vs Betweenness\n(size = community size)', 
                 fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    
    # Centrality comparison across communities
    ax = axes[1, 1]
    x = np.arange(len(communities))
    width = 0.2
    
    # Normalize for visualization
    for i, (name, values) in enumerate(comm_df[['degree', 'betweenness', 'eigenvector']].items()):
        if values.max() > 0:
            norm_values = values / values.max()
            ax.bar(x + i*width, norm_values, width, label=name, alpha=0.7)
    
    ax.set_xlabel('Community', fontsize=11)
    ax.set_ylabel('Normalized Centrality', fontsize=11)
    ax.set_title('Centrality Profiles by Community', fontsize=12, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('community_centrality_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Saved: community_centrality_analysis.png")

---
## 7. Export Results

In [ ]:
# Create comprehensive statistics DataFrame
export_data = {
    'edge_index': range(G_edges.vcount()),
    'edge_name': G_edges.vs['name'] if 'name' in G_edges.vs.attributes() else [f'Edge_{i}' for i in range(G_edges.vcount())],
    'degree': centralities['degree'],
    'betweenness': centralities['betweenness'],
    'closeness': centralities['closeness'],
    'eigenvector': centralities['eigenvector'],
    'pagerank': centralities['pagerank'],
    'composite_score': composite_score,
    'is_hub': is_hub
}

# Add community if available
if results:
    export_data['community'] = communities.membership

# Add original weight if available
if 'original_weight' in G_edges.vs.attributes():
    export_data['original_weight'] = G_edges.vs['original_weight']

# Create DataFrame and sort by composite score
results_df = pd.DataFrame(export_data)
results_df = results_df.sort_values('composite_score', ascending=False)

# Save to CSV
results_df.to_csv('centrality_hub_community_analysis.csv', index=False)
print("✓ Saved: centrality_hub_community_analysis.csv")

# Save updated network with all attributes
G_edges.write_graphml('edge_network_analyzed.graphml')
print("✓ Saved: edge_network_analyzed.graphml")

# Save community assignments separately
if results:
    comm_assignments = pd.DataFrame({
        'edge_index': range(G_edges.vcount()),
        'edge_name': G_edges.vs['name'] if 'name' in G_edges.vs.attributes() else [f'Edge_{i}' for i in range(G_edges.vcount())],
        'community': communities.membership
    })
    comm_assignments.to_csv('edge_community_assignments.csv', index=False)
    print("✓ Saved: edge_community_assignments.csv")

# Save summary report
with open('analysis_report.txt', 'w') as f:
    f.write("="*70 + "\n")
    f.write("EDGE NETWORK ANALYSIS REPORT\n")
    f.write("Centrality, Hubs, and Communities\n")
    f.write("="*70 + "\n\n")
    
    f.write("NETWORK PROPERTIES\n")
    f.write("-"*70 + "\n")
    f.write(f"Vertices (edges): {G_edges.vcount()}\n")
    f.write(f"Edges (edge-edge connections): {G_edges.ecount()}\n")
    f.write(f"Average degree: {np.mean(G_edges.degree()):.2f}\n")
    f.write(f"Density: {G_edges.density():.4f}\n\n")
    
    f.write("CENTRALITY STATISTICS\n")
    f.write("-"*70 + "\n")
    for name, values in centralities.items():
        f.write(f"{name.capitalize():15s}: mean={values.mean():.4f}, std={values.std():.4f}, "
                f"min={values.min():.4f}, max={values.max():.4f}\n")
    
    f.write(f"\nHUB EDGES\n")
    f.write("-"*70 + "\n")
    f.write(f"Number of hub edges: {n_hubs} ({100*n_hubs/G_edges.vcount():.1f}%)\n")
    f.write(f"Hub definition: Top 10% in both degree and betweenness\n")
    
    if results:
        f.write(f"\nCOMMUNITY STRUCTURE\n")
        f.write("-"*70 + "\n")
        f.write(f"Method: {best_method[0]}\n")
        f.write(f"Number of communities: {len(communities)}\n")
        f.write(f"Modularity: {communities.modularity:.4f}\n")
        f.write(f"Community sizes: {[len(c) for c in communities]}\n")

print("✓ Saved: analysis_report.txt")

print("\n" + "="*70)
print("ALL RESULTS EXPORTED")
print("="*70)
print("\nGenerated files:")
print("  • centrality_hub_community_analysis.csv - Complete results")
print("  • edge_network_analyzed.graphml - Network with all attributes")
print("  • edge_community_assignments.csv - Community memberships")
print("  • analysis_report.txt - Summary report")
print("  • Multiple PNG figures")

---
## 8. Summary

In [ ]:
print("="*70)
print("ANALYSIS SUMMARY")
print("="*70)

print(f"\nNetwork: {G_edges.vcount()} edges, {G_edges.ecount()} connections")
print(f"\nCentrality Measures Computed:")
for name in centralities.keys():
    print(f"  • {name.capitalize()}")

print(f"\nHub Edges: {n_hubs} ({100*n_hubs/G_edges.vcount():.1f}%)")

if results:
    print(f"\nCommunities: {len(communities)} (modularity = {communities.modularity:.4f})")

print("\nTop 5 Edges Overall (by composite score):")
for idx in results_df.head(5).index:
    print(f"  • {results_df.loc[idx, 'edge_name'][:60]}")

print("\n" + "="*70)
print("ANALYSIS COMPLETE")
print("="*70)